In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Works whether the notebook runs from the repo root or from logistical_regression_analysis/.
FOLDER = next(p for p in [Path('logistical_regression_analysis'), Path('.')] if (p / 'final_data_steven.csv').exists())
raw = pd.read_csv(FOLDER / 'final_data_steven.csv')
assert raw.shape == (8984, 30) and raw['respondent_id'].is_unique
print('Loaded final_data_steven.csv:', raw.shape)


Loaded final_data_steven.csv: (8984, 30)


In [2]:
# NLSY97 stores missing answers as negative codes. Only these five mean "missing":
#   -1 refused, -2 don't know, -3 invalid skip, -4 valid skip (question not asked), -5 not interviewed
MISSING_CODES = [-1, -2, -3, -4, -5]

def to_missing(column):
    # turns exactly these five codes into `NaN`. It doesn't touch other negative numbers, 
    # because some are real: household net worth can be negative (debt).
    return column.mask(column.isin(MISSING_CODES))

clean = pd.DataFrame({'respondent_id': raw['respondent_id']})


Treatment Z and outcome Y


In [3]:
# Treatment Z and outcome Y, exactly as the team defined them (Albert's and Nicole's rules).
clean['Z'] = raw['timely_college']                 # 1 = first college entry at ages 18-20, 0 = not
clean['Y'] = raw['violent_victimization_18_24']    # 1 = violent victimization at ages 18-24 (see team notes)
print('Z:', clean['Z'].value_counts(dropna=False).to_dict())
print('Y:', clean['Y'].value_counts(dropna=False).to_dict())


Z: {0.0: 4688, 1.0: 4158, nan: 138}
Y: {0.0: 7952, 1.0: 520, nan: 512}


Demographics

Neither variable has missing values.
- `female`: 1 if female, 0 if male.
- `race`: Five categories.

In [4]:
clean['female'] = (raw['sex'] == 2).astype(float)  # sex: 1 = male, 2 = female (never missing)
clean['race'] = raw['race_ethnicity']              # White, Black, Hispanic, Asian, Other (never missing)
print(clean['race'].value_counts().to_dict())


{'White': 4413, 'Black': 2334, 'Hispanic': 1899, 'Other': 182, 'Asian': 156}


Family background (1997)

- `networth_10k`: household net worth in units of $10,000. Topcoded at $600,000; negative values are kept.
- `hh_size`: number of people in the household.
- `mother_educ`, `father_educ`: highest grade completed, 0–20 years. Code 95 ("ungraded") becomes missing.
- `hard_times`: parent report that the youth ever lived without water or electricity, or in a shelter.
- `parent_prison`: 1 if either parent served a prison sentence before the respondent was 16, 0 if neither did, missing otherwise. It was asked in 2013, but it describes childhood.

In [5]:
# Household net worth (1997, parent report). Negative values are real debts and are kept.
# Topcoded at $600,000. Divided by 10,000 so the unit is "$10,000".
clean['networth_10k'] = to_missing(raw['household_net_worth']) / 10000

clean['hh_size'] = raw['household_size']           # number of people in the household, 1997 (never missing)

# Parents' highest grade completed (1997). Valid years are 0-20; code 95 means "ungraded", so it is
# set to missing along with the NLS missing codes.
for parent in ['mother', 'father']:
    years = raw[f'biological_{parent}_education']
    clean[f'{parent}_educ'] = years.where(years.between(0, 20))

# Parent report: has the youth ever lived through hard times (no water/electricity, shelter)? 1 = yes.
clean['hard_times'] = to_missing(raw['r_ever_live_through_hard_times'])

# Mother / father served a prison sentence before the respondent turned 16 (asked in 2013).
# Combined: 1 if either parent did, 0 if both did not, missing otherwise.
mom = to_missing(raw['mother_serve_prison_sentence'])
dad = to_missing(raw['father_serve_prison_sentence'])
clean['parent_prison'] = np.where((mom == 1) | (dad == 1), 1.0,
                                  np.where((mom == 0) & (dad == 0), 0.0, np.nan))


Academic preparation and school (1997)

- `asvab`: the stored value has 3 implied decimals, so it's divided by 1,000 to give a 0–100 percentile.
- The school questions were only asked of youth enrolled in school, so a few are missing:
  - `days_absent`, `threatened_school` and `fight_school` are **counts** for the 1997 school year.
  - `teachers_good` and `discipline_fair` run from 1 = strongly agree to 4 = strongly disagree.
  - The peer items run from 1 = almost none to 5 = almost all.

In [6]:
# ASVAB math/verbal percentile. Stored with 3 implied decimals (0-100,000), so divide by 1,000 -> 0-100.
asvab = to_missing(raw['asvab_math_verbal_score_percent'])
clean['asvab'] = asvab / 1000

# 1997 school questions (only asked of youth enrolled in K-12 school):
clean['days_absent'] = to_missing(raw['days_r_absent_from_school'])          # days absent, fall term
clean['teachers_good'] = to_missing(raw['teachers_good'])                    # 1 strongly agree ... 4 strongly disagree
clean['discipline_fair'] = to_missing(raw['discipline_is_fair_agree_disagree'])  # same 1-4 scale
clean['threatened_school'] = to_missing(raw['r_threatened_to_be_hurt_at_school'])  # times threatened at school
clean['fight_school'] = to_missing(raw['r_in_a_fight_at_school'])                  # times in a fight at school

# Share of peers who smoke / get drunk monthly / belong to a gang:
# 1 almost none, 2 about 25%, 3 about half, 4 about 75%, 5 almost all
clean['peers_smoke'] = to_missing(raw['percent_of_peers_who_smoke'])
clean['peers_drunk'] = to_missing(raw['percent_of_peers_who_get_drunk_1_times_a_month'])
clean['peers_gang'] = to_missing(raw['percent_of_peers_belong_to_a_gang'])


`earliest_arrest_date` is the month and year of a person's **first** arrest at any point up to 2024. Arrests after 18 can happen after college entry, so the date can't be used as-is: we keep only whether the first arrest came **before the month the person turned 18**.

- Code −4 means never arrested, so 0.
- Code −3 (invalid skip) becomes missing.
- An arrest in the 18th-birthday month itself counts as 0, because the exact birth day isn't public.

In [7]:
# Bullied repeatedly before age 12 (1 = yes).
clean['bullied_before_12'] = to_missing(raw['bullying_victim_before_age_12'])
# Bullied at ages 12-18
clean['bullied_12_18'] = raw['r_victim_of_bullying_12_18_years']

# Arrested before age 18.
# earliest_arrest_date is the respondent's FIRST arrest at any time up to 2024, coded YYYYMM
# (e.g. 199808 = August 1998). -4 = never arrested, -3 = invalid skip (missing).
# Compare the arrest month with the month of the 18th birthday (the exact birth day is not public):
#   arrest month earlier than the 18th-birthday month -> 1 (definitely before 18)
#   arrest in or after the 18th-birthday month, or never arrested -> 0
arrest = raw['earliest_arrest_date']
arrest_month_index = (arrest // 100) * 12 + (arrest % 100)                   # year*12 + month
birthday_18_index = (raw['birth_year'] + 18) * 12 + raw['birth_month']        # year*12 + month of 18th birthday
clean['arrested_before_18'] = np.select(
    [arrest == -4,                                               # never arrested
     (arrest > 0) & (arrest_month_index < birthday_18_index),   # first arrest before 18
     arrest > 0],                                                # first arrest at 18 or later
    [0.0, 1.0, 0.0],
    default=np.nan)                                              # -3 invalid skip
print('arrested_before_18:', clean['arrested_before_18'].value_counts(dropna=False).to_dict())
print('   first arrest in the 18th-birthday month (counted as 0):',
      int(((arrest > 0) & (arrest_month_index == birthday_18_index)).sum()))


arrested_before_18: {0.0: 7370, 1.0: 1576, nan: 38}
   first arrest in the 18th-birthday month (counted as 0): 35


Columns not used

In [8]:
pd.set_option('display.max_colwidth', None)  
dropped = pd.DataFrame([
    ('earliest_arrest_date', 'Covers arrests up to 2024, after college. Replaced by arrested_before_18.'),
    ('total_number_of_incarcerations', 'Counts incarcerations up to 2024, including after college; no dates to limit it to before 18.'),
    ('what_year_first_arrest_arrest_10', f"Missing for {int(raw['what_year_first_arrest_arrest_10'].isin(MISSING_CODES).sum()):,} of 8,984."),
    ('percent_chance_r_victim_of_violent_crime_by_next_year', f"Only asked of youth 15+ in 1997: missing for {int(raw['percent_chance_r_victim_of_violent_crime_by_next_year'].isin(MISSING_CODES).sum()):,}."),
    ('percent_of_peers_who_have_had_sex', f"Asked of a subset: missing for {int(raw['percent_of_peers_who_have_had_sex'].isin(MISSING_CODES).sum()):,}."),
    ('birth_month, birth_year', 'Used only to build arrested_before_18; birth year is not a covariate.'),
], columns=['column', 'reason'])
display(dropped)


,column,reason
0,earliest_arrest_date,"Covers arrests up to 2024, after college. Replaced by arrested_before_18."
1,total_number_of_incarcerations,"Counts incarcerations up to 2024, including after college; no dates to limit it to before 18."
2,what_year_first_arrest_arrest_10,"Missing for 8,982 of 8,984."
3,percent_chance_r_victim_of_violent_crime_by_next_year,"Only asked of youth 15+ in 1997: missing for 5,498."
4,percent_of_peers_who_have_had_sex,"Asked of a subset: missing for 5,019."
5,"birth_month, birth_year",Used only to build arrested_before_18; birth year is not a covariate.


## Summary of the cleaned covariates

In [9]:
COVARIATES = ['female', 'race', 'networth_10k', 'hh_size', 'mother_educ', 'father_educ', 'hard_times',
              'parent_prison', 'asvab', 'days_absent', 'teachers_good', 'discipline_fair', 'threatened_school',
              'fight_school', 'peers_smoke', 'peers_drunk', 'peers_gang', 'bullied_before_12', 'bullied_12_18',
              'arrested_before_18']
summary = pd.DataFrame({
    'missing': clean[COVARIATES].isna().sum(),
    'missing %': (100 * clean[COVARIATES].isna().mean()).round(1),
    'min': [clean[c].min() if c != 'race' else '' for c in COVARIATES],
    'max': [clean[c].max() if c != 'race' else '' for c in COVARIATES],
    'mean': [round(clean[c].mean(), 3) if c != 'race' else '' for c in COVARIATES],
})
display(summary)

# Check: no NLS missing codes are left. After cleaning, only net worth can be negative (real debts).
numeric = [c for c in COVARIATES if c != 'race']
assert not clean[[c for c in numeric if c != 'networth_10k']].lt(0).any().any()
print('Columns in the cleaned file:', len(clean.columns))


,missing,missing %,min,max,mean
female,0,0.0,0.0,1.0,0.488
race,0,0.0,,,
networth_10k,2365,26.3,-93.5251,60.0,9.018
hh_size,0,0.0,1,16,4.549
mother_educ,694,7.7,1.0,20.0,12.438
father_educ,1864,20.7,1.0,20.0,12.564
hard_times,1109,12.3,0.0,1.0,0.055
parent_prison,2473,27.5,0.0,1.0,0.076
asvab,1891,21.0,0.0,100.0,45.317
days_absent,297,3.3,0.0,200.0,4.889


Columns in the cleaned file: 23


## Save

In [10]:
clean.to_csv(FOLDER / 'clean_data.csv', index=False)
print('Saved', FOLDER / 'clean_data.csv', clean.shape)


Saved clean_data.csv (8984, 23)
